## 💬 Voltagent Example with Tool Call

This little script is basically your own mini **AI-powered Pokédex**.
You ask it something like *“Tell me about Pikachu”*, and it will:

1. Think about your question using an AI model (GitHub’s GPT-4o-mini),
2. Realize it needs real-world Pokémon data,
3. Call your custom Pokémon tool that fetches info from the **PokéAPI**,
4. And finally respond with a short summary — which gets printed to your console.

---

## 🔧 Breaking It Down

### 1. You connect to GitHub’s AI models

```ts
const githubModels = createOpenAICompatible({...})
```

This sets up access to GitHub’s hosted AI models (like GPT-4).
You’re telling the AI where to send requests and passing your GitHub token for authentication.

---

### 2. You give the AI a “Pokémon tool”

```ts
const getPokemon = tool({...})
```

This tool is like a special ability the AI can use whenever it needs facts about a Pokémon.

* It expects one input: `{ name: "pikachu" }`
* It fetches real data from `https://pokeapi.co/api/v2/pokemon/...`
* It returns things like ID, height, weight, and types

So if the AI needs real data — it doesn’t make it up — it calls this tool.

---

### 3. You create the AI Agent

```ts
const agent = new Agent({
  name: "pokemon-agent",
  instructions: "When asked about a Pokémon, call get_pokemon...",
  model: githubModels("openai/gpt-4o-mini"),
  tools: [getPokemon],
});
```

This is where everything comes together.

You’re basically telling the AI:

> “Your name is Pokémon Agent. If I ask you about a Pokémon, use the tool I gave you — don’t just guess.”

---

### 4. You actually talk to it

```ts
const result = await agent.generateText("Tell me about Pikachu.");
console.log(result.text);
```

This is the final step:

* You ask a question
* The AI thinks
* It decides to use the Pokémon tool
* The tool fetches real data from PokéAPI
* The AI summarizes and gives it back to you
* You print the answer

---

## ✅ Example of What You’ll See

```
Pikachu (ID: 25)
• Type: electric
• Height: 4
• Weight: 60
```




In [6]:
// deno run --allow-env --allow-net pokemon-test.ts
// ENV: GITHUB_TOKEN (token with `models:read`)

import { Agent, tool } from "npm:@voltagent/core@^1";
import { createOpenAICompatible } from "npm:@ai-sdk/openai-compatible@^1";
import { z } from "npm:zod@^3";

// 1) GitHub Models (OpenAI-compatible)
const githubModels = createOpenAICompatible({
  name: "github-models",
  baseURL: "https://models.github.ai/inference",
  headers: {
    Authorization: `Bearer ${Deno.env.get("GITHUB_TOKEN") ?? ""}`,
    "X-GitHub-Api-Version": "2022-11-28",
    "Content-Type": "application/json",
  },
});

// 2) Pokémon tool — NOTE: `parameters` is a Zod schema
const getPokemon = tool({
  name: "get_pokemon",
  description: "Get basic Pokémon data by name (id, types, height, weight).",
  parameters: z
    .object({
      name: z
        .string()
        .min(1)
        .describe("Name of the Pokémon (e.g., 'pikachu')."),
    })
    .describe("Arguments for the get_pokemon tool."),
  execute: async ({ name }) => {
    const url = `https://pokeapi.co/api/v2/pokemon/${encodeURIComponent(
      name.toLowerCase(),
    )}`;
    const res = await fetch(url);
    if (!res.ok) return { error: `Not found: ${name}` };
    const data = await res.json();
    return {
      id: data.id,
      name: data.name,
      types: (data.types ?? []).map((t: any) => t.type?.name),
      height: data.height,
      weight: data.weight,
    };
  },
});

// 3) Agent with tool
const agent = new Agent({
  name: "pokemon-agent",
  instructions:
    "When asked about a Pokémon, call get_pokemon and reply with a short bullet list.",
  model: githubModels("openai/gpt-4o-mini"), // or "openai/gpt-4.1-mini"
  tools: [getPokemon],
});

// 4) Generate once and print result text
const result = await agent.generateText("Tell me about Pikachu.");
console.log(result.text);


Here's some information about Pikachu:

- **ID**: 25
- **Type**: Electric
- **Height**: 0.4 meters (1'04")
- **Weight**: 6.0 kg (13.2 lbs)
